In [1]:
# Cell 1
import sys
import json
import pandas as pd
from collections import Counter
 
sys.path.append('../')
from src.compliance.alignment_engine import AlignmentEngine
from src.graph_rag.retriever         import GraphRAGRetriever
from src.graph_rag.explainer         import LLMExplainer
from src.graph_rag.report_generator  import ComplianceReportGenerator

e:\graph-rag-compliance\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# Cell 2
engine    = AlignmentEngine(env_path='../.env')
retriever = GraphRAGRetriever(engine.driver, k_hop=2)
explainer = LLMExplainer(model='claude-haiku-4-5-20251001')
generator = ComplianceReportGenerator(engine, retriever, explainer)

Regulation loaded: 13 activities, 6 sequences, 4 conditions
Loading embedding model...
Embedding model loaded
LLM Explainer ready: claude-haiku-4-5-20251001


In [3]:
# Cell 3
report_skip = generator.generate('Application_1858876732')
report_skip.print_report()


COMPLIANCE REPORT
Mã hồ sơ     : Application_1858876732
Loại hồ sơ   : New credit
Mục đích vay : Home improvement
Số tiền vay  : 25,000 EUR
Số sự kiện   : 56
Tuân thủ     : ❌ Không
Fitness score: 0.40

Vi phạm (2):
------------------------------------------------------------

[1] HIGH — SKIP
     Điều khoản : Article 5(1)
     Mô tả      : Bước bắt buộc 'O_Sent (mail and online)' bị bỏ qua

     Graph RAG context:
     Query: O_Sent (mail and online)
     Nodes retrieved: O_Sent (mail and online), A_Complete

     Giải thích (LLM):
     TÓM TẮT: Bỏ qua bước gửi thông tin tín dụng cho khách hàng qua thư/online theo yêu cầu bắt buộc.
     LÝ DO: Điều 5(1) Chỉ thị 2008/48/EC yêu cầu nhà cung cấp phải gửi các điều khoản và thông tin tín dụng cho người tiêu dùng trước khi ký kết hợp đồng. Trace của hồ sơ 1858876732 không chứa hoạt động O_Sent, vi phạm nghĩa vụ công khai thông tin bắt buộc này.
     KHẮC PHỤC: Bổ sung bước gửi thông tin tín dụng hoàn chỉnh qua mail/online cho khách hàng trư

In [ ]:
# Cell debug — kiểm tra API key
import os
from dotenv import load_dotenv

load_dotenv('../.env')

key = os.getenv("ANTHROPIC_API_KEY")
if key:
    print(f"API key tìm thấy: {key[:15]}...")
else:
    print("KHÔNG tìm thấy API key")

In [ ]:
# Cell 4 — Test temporal violation
report_temporal = generator.generate('Application_1000610355')
report_temporal.print_report()
 

In [4]:
# Cell 5 — 1 case tuân thủ từ compliance_results.json
with open('../data/processed/compliance_results.json',
          encoding='utf-8') as f:
    all_results = json.load(f)
 
compliant_cases = [
    r['case_id'] for r in all_results if r['is_compliant']
]
print(f"Số case tuân thủ trong file: {len(compliant_cases):,}")
 
sample_compliant = compliant_cases[0]
report_ok = generator.generate(sample_compliant)
report_ok.print_report()

Số case tuân thủ trong file: 24,221

COMPLIANCE REPORT
Mã hồ sơ     : Application_1000158214
Loại hồ sơ   : New credit
Mục đích vay : Home improvement
Số tiền vay  : 12,500 EUR
Số sự kiện   : 25
Tuân thủ     : ✅ Có
Fitness score: 1.00

✅ Không phát hiện vi phạm.


In [6]:
# Cell 6 — 20 case vi phạm để demo batch
violated_cases = [
    r['case_id'] for r in all_results
    if not r['is_compliant']
][:20]
 
print(f"Sẽ generate report cho {len(violated_cases)} case vi phạm...")
print(f"Case IDs: {violated_cases[:20]} ...")

Sẽ generate report cho 20 case vi phạm...
Case IDs: ['Application_1000610355', 'Application_1000647090', 'Application_1000856802', 'Application_1001114274', 'Application_1001484676', 'Application_1002013470', 'Application_1002051388', 'Application_1002129113', 'Application_1002626536', 'Application_1002961837', 'Application_1003257895', 'Application_1003444518', 'Application_1003634925', 'Application_1004667877', 'Application_1004718670', 'Application_1005228320', 'Application_1005677212', 'Application_1005878170', 'Application_1006214165', 'Application_1007290684'] ...


In [7]:
#cell 7 Batch generate
batch_reports = generator.generate_batch(
    case_ids  = violated_cases,
    save_path = '../data/processed/sample_explained_reports.json',
    verbose   = True
)
 
compliant_count = sum(1 for r in batch_reports if r.is_compliant)
print(f"\nTổng      : {len(batch_reports)}")
print(f"Tuân thủ  : {compliant_count}")
print(f"Vi phạm   : {len(batch_reports) - compliant_count}")

  5/20 generated
  10/20 generated
  15/20 generated
  20/20 generated
Đã lưu 20 report: ../data/processed/sample_explained_reports.json

Tổng      : 20
Tuân thủ  : 0
Vi phạm   : 20


In [8]:
# Cell 8 — Thống kê theo type / severity / article 
vtype_counter    = Counter()
severity_counter = Counter()
article_counter  = Counter()
 
for r in batch_reports:
    for ev in r.explained_violations:
        v = ev.violation
        vtype_counter[v['type']]          += 1
        severity_counter[v['severity']]   += 1
        article_counter[v['article_ref']] += 1
 
print("Phân loại vi phạm theo type:")
for vtype, cnt in vtype_counter.most_common():
    print(f"  {vtype:<20}: {cnt}")
 
print("\nPhân loại theo mức độ:")
for sev, cnt in severity_counter.most_common():
    print(f"  {sev:<10}: {cnt}")
 
print("\nPhân loại theo điều khoản:")
for art, cnt in article_counter.most_common():
    print(f"  {art:<20}: {cnt}")
 

Phân loại vi phạm theo type:
  temporal            : 21

Phân loại theo mức độ:
  medium    : 21

Phân loại theo điều khoản:
  Article 14(1)       : 15
  Article 7(1)        : 5
  Article 5(1)        : 1


In [9]:
#  Cell 9 — Phân phối fitness score 
fitness_scores = [r.fitness_score for r in batch_reports]
df_batch = pd.DataFrame({
    'case_id'         : [r.case_id          for r in batch_reports],
    'is_compliant'    : [r.is_compliant      for r in batch_reports],
    'fitness_score'   : fitness_scores,
    'num_violations'  : [r.num_violations    for r in batch_reports],
    'application_type': [r.application_type  for r in batch_reports],
    'loan_goal'       : [r.loan_goal         for r in batch_reports],
    'requested_amount': [r.requested_amount  for r in batch_reports],
})
 
print("Fitness score — thống kê:")
print(df_batch['fitness_score'].describe().round(3))
 
print("\nSố vi phạm trên mỗi case:")
print(df_batch['num_violations'].value_counts().sort_index())

Fitness score — thống kê:
count    20.000
mean      0.842
std       0.034
min       0.700
25%       0.850
50%       0.850
75%       0.850
max       0.850
Name: fitness_score, dtype: float64

Số vi phạm trên mỗi case:
num_violations
1    19
2     1
Name: count, dtype: int64


In [ ]:
#  Cell 10 — Case có nhiều vi phạm nhất 
most_violated = max(batch_reports, key=lambda r: r.num_violations)
print(f"Case có nhiều vi phạm nhất: {most_violated.case_id}")
print(f"Số vi phạm: {most_violated.num_violations}")
print()
most_violated.print_report()
 

Case có nhiều vi phạm nhất: Application_1001114274
Số vi phạm: 2


COMPLIANCE REPORT
Mã hồ sơ     : Application_1001114274
Loại hồ sơ   : New credit
Mục đích vay : Car
Số tiền vay  : 25,000 EUR
Số sự kiện   : 69
Tuân thủ     : ❌ Không
Fitness score: 0.70

Vi phạm (2):
------------------------------------------------------------

[1] MEDIUM — TEMPORAL
     Điều khoản : Article 14(1)
     Mô tả      : Rút lui sau 28 ngày, vượt quá thời hạn 14 ngày

     Graph RAG context:
     Query: Article 14(1) time limit deadline
     Nodes retrieved: A_Create Application, A_Submitted, A_Concept

     Giải thích (LLM):
     TÓM TẮT: Đơn vị cấp tín dụng rút lui sau 28 ngày vượt quá thời hạn 14 ngày quy định tại Article 14(1) Chỉ thị 2008/48/EC.
     LÝ DO: Article 14(1) yêu cầu nhà cung cấp tín dụng phải rút lui trong vòng 14 ngày kể từ khi gửi cho người tiêu dùng. Trong trường hợp này, OSent ngày 15/12/2016 nhưng OReturned ngày 13/01/2017, tức 28 ngày sau, vượt quá 14 ngày cho phép. Hành vi này vi ph

In [ ]:
# Cell 11 — Detail retrieval + explanation của một vi phạm cụ thể
if batch_reports and batch_reports[0].explained_violations:
    first_ev = batch_reports[0].explained_violations[0]
 
    print("=" * 50)
    print("GRAPH RAG RETRIEVAL DETAIL")
    print("=" * 50)
    print(f"\nVi phạm: {first_ev.violation['description']}")
    print(f"Query  : {first_ev.context['query']}")
 
    print("\nEntry nodes (vector search):")
    for node in first_ev.context.get('entry_nodes', []):
        sim = node.get('similarity', 0)
        print(f"  [{sim:.3f}] {node.get('name','')} "
              f"— {node.get('article_ref','')}")
 
    print("\nRegulation context (đưa vào LLM):")
    print(first_ev.context.get('regulation_text', ''))
 
    print("\nGiải thích LLM (plain text):")
    print(first_ev.explanation)

GRAPH RAG RETRIEVAL DETAIL

Vi phạm: Rút lui sau 18 ngày, vượt quá thời hạn 14 ngày
Query  : Article 14(1) time limit deadline

Entry nodes (vector search):
  [1.000] A_Create Application — Article 8(1)
  [1.000] A_Submitted — Article 8(1)
  [1.000] A_Concept — Article 8(1)

Regulation context (đưa vào LLM):
REGULATORY CONTEXT (Article 14(1)):

Activity    : A_Create Application
Description : Consumer submits credit application
Legal basis : Article 8(1)
Required    : True
Document    : Directive 2008/48/EC on Credit Agreements for Consumers
Followed by : A_Concept
Performed by: Creditor System

Activity    : A_Submitted
Description : Application formally submitted to creditor
Legal basis : Article 8(1)
Required    : False
Document    : Directive 2008/48/EC on Credit Agreements for Consumers
Performed by: Creditor System

Activity    : A_Concept
Description : Initial creditworthiness concept assessment
Legal basis : Article 8(1)
Required    : True
Document    : Directive 2008/48/EC on 

In [12]:
# Cell 12 — Thống kê vi phạm theo mục đích vay
rows = []
for r in batch_reports:
    for ev in r.explained_violations:
        rows.append({
            'case_id'         : r.case_id,
            'loan_goal'       : r.loan_goal,
            'application_type': r.application_type,
            'requested_amount': r.requested_amount,
            'violation_type'  : ev.violation['type'],
            'severity'        : ev.violation['severity'],
            'article_ref'     : ev.violation['article_ref'],
            'fitness_score'   : r.fitness_score,
        })
 
if rows:
    df_violations = pd.DataFrame(rows)
    print("Vi phạm theo mục đích vay:")
    print(
        df_violations.groupby(['loan_goal', 'violation_type'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .to_string(index=False)
    )
else:
    print("Không có vi phạm nào trong batch.")

Vi phạm theo mục đích vay:
             loan_goal violation_type  count
Existing loan takeover       temporal      6
      Home improvement       temporal      5
                   Car       temporal      4
Other, see explanation       temporal      3
         Not speficied       temporal      2
               Unknown       temporal      1


In [13]:
#  Cell 13 — Lưu batch report ra file JSON 
import os
 
output_path = '../data/processed/sample_explained_reports.json'
os.makedirs('../data/processed', exist_ok=True)
 
output_data = [r.to_dict() for r in batch_reports]
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)
 
print(f"Đã lưu {len(output_data)} report: {output_path}")
print(f"\nCấu trúc 1 record:")
if output_data:
    sample = output_data[0]
    print(f"  Keys: {list(sample.keys())}")
    if sample.get('violations'):
        v = sample['violations'][0]
        print(f"  Violation keys: {list(v.keys())}")

Đã lưu 20 report: ../data/processed/sample_explained_reports.json

Cấu trúc 1 record:
  Keys: ['case_id', 'application_type', 'loan_goal', 'requested_amount', 'num_events', 'is_compliant', 'fitness_score', 'violation_summary', 'violations']
  Violation keys: ['violation', 'context', 'explanation']


In [14]:
#cell 14 — Tổng kết
print("=" * 60)
print("TỔNG KẾT NOTEBOOK 06 — GRAPH RAG EXPLAIN")
print("=" * 60)
print(f"\nBatch size       : {len(batch_reports)} case vi phạm")
 
total_violations = sum(r.num_violations for r in batch_reports)
print(f"Tổng vi phạm     : {total_violations}")
print(f"TB vi phạm/case  : {total_violations/len(batch_reports):.1f}")
 
avg_fitness = sum(r.fitness_score for r in batch_reports) / len(batch_reports)
print(f"Fitness TB       : {avg_fitness:.3f}")
 
print(f"\nLLM model        : claude-haiku-4-5-20251001")
print(f"Embedding model  : paraphrase-multilingual-MiniLM-L12-v2")
print(f"k-hop traversal  : 2")
print(f"\nOutput saved     : data/processed/sample_explained_reports.json")
print("\n→ Chạy notebook 07 để so sánh với Token-based Replay baseline.")

TỔNG KẾT NOTEBOOK 06 — GRAPH RAG EXPLAIN

Batch size       : 20 case vi phạm
Tổng vi phạm     : 21
TB vi phạm/case  : 1.1
Fitness TB       : 0.842

LLM model        : claude-haiku-4-5-20251001
Embedding model  : paraphrase-multilingual-MiniLM-L12-v2
k-hop traversal  : 2

Output saved     : data/processed/sample_explained_reports.json

→ Chạy notebook 07 để so sánh với Token-based Replay baseline.
